# STAT 4845/5845 Lecture 6: areal data, median 2024 single family sale price in the Connecticut towns

Runtime > Change runtime type > R before running.

## Setup

Areal data come as polygons with attributes. The `sf` package reads, stores and plots them. Colab's R runtime may not ship it, so install it once per session (a few minutes).

In [ ]:
if (!requireNamespace("sf", quietly = TRUE)) install.packages("sf")

library(sf)

## Data

One file, shared on Google Drive: the 169 Connecticut towns as polygons, with the median 2024 single family sale price (dollars), the number of sales, the town centroid, and the ACS 2020-2024 median household income, population, area and density.

In [ ]:
download.file("https://drive.google.com/uc?export=download&id=1uAfgLYtNFgTvUz1vL4kEESPlhhUL6mnF", "ct_towns_2024.rds", mode = "wb")

ct <- readRDS("ct_towns_2024.rds")

## Explore

An `sf` object is a data frame with one row per region and a geometry column holding the polygon.

In [ ]:
class(ct)

nrow(ct)  # number of towns

names(ct)

ct  # the first rows, the geometry column at the end

st_crs(ct)$Name  # coordinate reference system, longitude and latitude

In [ ]:
st_geometry(ct)[[1]]  # the polygon of the first town, a list of boundary coordinates

summary(ct$median_price)  # 18 towns without a listed 2024 sale

ct$town[is.na(ct$median_price)]

## Map the towns

The polygons alone, then each attribute as a choropleth map.

In [ ]:
plot(st_geometry(ct), border = "grey40")

text(ct$lon, ct$lat, ct$town, cex = 0.3)

In [ ]:
pal <- function(k) hcl.colors(k, "Viridis")

plot(ct["median_price"], main = "Median sale price 2024, dollars", pal = pal, border = "grey40")  # white: no sales data

In [ ]:
plot(ct["n_sales"], main = "Number of single family sales 2024", pal = pal, border = "grey40")

plot(ct["median_income"], main = "Median household income, dollars", pal = pal, border = "grey40")

plot(ct["density"], main = "Population density, people per km2", pal = pal, border = "grey40")

## Neighbors

Two towns are neighbors when their polygons touch, sharing an edge or a corner (queen contiguity). `st_touches` works this out from the geometry and returns, for each town, the row numbers of its neighbors.

In [ ]:
n <- nrow(ct)

nb <- st_touches(ct)  # queen neighbors: share an edge or corner

nb[[which(ct$town == "Tolland")]]  # the neighbors of Tolland, by row number
ct$town[nb[[which(ct$town == "Tolland")]]]

In [ ]:
plot(st_geometry(ct), border = "grey60")
for (i in 1:n) segments(ct$lon[i], ct$lat[i], ct$lon[nb[[i]]], ct$lat[nb[[i]]], col = "#004369")
points(ct$lon, ct$lat, pch = 16, cex = 0.5)

## Proximity matrix W

Build W entry by entry: w_ij = 1 if towns i and j are neighbors, 0 otherwise.

In [ ]:
W <- matrix(0, n, n)
for (i in 1:n) W[i, nb[[i]]] <- 1  # w_ij = 1 if towns i and j share a border

dim(W)

isSymmetric(W)

i <- which(ct$town == "Tolland")
W[i, ]  # the row of Tolland, mostly zeros
which(W[i, ] == 1)  # the same four neighbors as nb

table(rowSums(W))  # number of neighbors per town

ct$town[rowSums(W) == max(rowSums(W))]  # the town with the most neighbors

## Moran's I

Prices in thousand dollars. The 18 towns without a sale are left out: z holds the deviations from the mean for the 151 towns with data, and Wo is W restricted to those towns.

In [ ]:
y <- ct$median_price / 1000  # thousand dollars

obs <- !is.na(y)  # towns with a listed 2024 sale
z <- y[obs] - mean(y[obs])  # deviations from the mean
Wo <- W[obs, obs]

I <- (sum(obs) / sum(Wo)) * sum(z * (Wo %*% z)) / sum(z^2)  # Moran's I
I

-1 / (sum(obs) - 1)  # expected value under no spatial autocorrelation

## Permutation test

Shuffle the prices among the towns, W fixed, and recompute I each time.

In [ ]:
set.seed(1)
I_perm <- replicate(999, {
  zp <- sample(z)
  (sum(obs) / sum(Wo)) * sum(zp * (Wo %*% zp)) / sum(zp^2)
})

hist(I_perm, breaks = 40, col = "#79AFBA", border = "white", xlim = range(c(I_perm, I)), main = "Moran's I of 999 permutations", xlab = "I")
abline(v = I, lwd = 3, col = "#BE2D2D")

(sum(I_perm >= I) + 1) / (999 + 1)  # p value

## Local Moran's I and the LISA cluster map

I_i = n z_i (sum_j w_ij z_j) / sum_i z_i^2, one value per town. Its significance comes from a conditional permutation test: z_i stays fixed, its neighbors are drawn from the other towns.

In [ ]:
no <- sum(obs)
s2 <- sum(z^2)

I_local <- as.vector(no * z * (Wo %*% z) / s2)  # local Moran's I, one value per town

head(data.frame(town = ct$town[obs], z = round(z, 1), I_local = round(I_local, 2)), 10)

set.seed(1)
p_local <- sapply(1:no, function(i) {
  sims <- replicate(999, no * z[i] * sum(sample(z[-i], sum(Wo[i, ]))) / s2)  # neighbors drawn from the other towns
  (min(sum(sims >= I_local[i]), sum(sims <= I_local[i])) + 1) / (999 + 1)  # two sided
})

sum(p_local < 0.05)  # towns with a significant local I

In [ ]:
lag <- as.vector(Wo %*% z) / rowSums(Wo)  # average deviation of the neighbors

quad <- ifelse(z >= 0 & lag >= 0, "High-High", ifelse(z < 0 & lag < 0, "Low-Low", ifelse(z >= 0, "High-Low", "Low-High")))
cl <- ifelse(p_local < 0.05, quad, "Not significant")
table(cl)

ct$town[obs][cl %in% c("High-Low", "Low-High")]  # spatial outliers

lcols <- c("High-High" = "#BE2D2D", "Low-Low" = "#004369", "High-Low" = "#F2A9A9", "Low-High" = "#9CC3D5", "Not significant" = "#EDEDED")
fill <- rep("#7A7A7A", n)  # no data
fill[obs] <- lcols[cl]
plot(st_geometry(ct), col = fill, border = "grey45", main = "LISA cluster map, p < 0.05")
legend("bottomright", legend = c(names(lcols), "No data"), fill = c(lcols, "#7A7A7A"), border = "grey45", bty = "n", cex = 0.8)

## Moran scatterplot

Horizontal axis: z_i. Vertical axis: the average z_j over the neighbors of town i. The slope of the fitted line is Moran's I with row standardized weights.

In [ ]:
plot(z, lag, pch = 16, col = lcols[quad], xlab = "Deviation from the mean, z (thousand dollars)", ylab = "Average deviation of the neighbors")
abline(h = 0, v = 0, col = "grey60")
abline(lm(lag ~ z), lwd = 3, col = "#D55E00")
text(max(z), c(max(lag), min(lag)), c("High-High", "High-Low"), adj = 1, cex = 0.9)
text(min(z), c(min(lag) / 2, max(lag)), c("Low-Low", "Low-High"), adj = 0, cex = 0.9)

coef(lm(lag ~ z))[2]  # the slope

Wso <- Wo / rowSums(Wo)  # row standardized
(sum(obs) / sum(Wso)) * sum(z * (Wso %*% z)) / sum(z^2)  # Moran's I with row standardized weights